# 03 — Heston–Merton

**What this notebook is:** a complete start-to-finish guide for Heston stochastic volatility **plus** Merton jumps, plus an interactive Monte Carlo playground.

Same structure as [`01_gbm.ipynb`](01_gbm.ipynb), [`02_merton.ipynb`](02_merton.ipynb), and [`04_garch_merton.ipynb`](04_garch_merton.ipynb):
model idea → workflow → how to estimate parameters → constant vs path-updating → playground.

**Theory handbook:** [`00_MODELS_EXPLAINED.ipynb`](00_MODELS_EXPLAINED.ipynb)

**Your project data (already prepared):**
- Prices: `../data/equity/prices_clean.csv`
- Log returns: `../data/equity/log_returns_all.csv`, `../data/equity/log_returns_by_regime.csv`
- Summary stats: `../data/equity/summary_stats.csv`

The playground at the bottom uses **synthetic** paths (sliders). The sections above it explain how you would use **real** historical data to choose those parameters.


## 1. Model idea (keep this mental picture)

Volatility is **itself random** (Heston) and the price can still jump (Merton). Instantaneous variance $v_t$ mean-reverts to a long-run level $\theta$.

### Continuous (differential) equations

**Variance:**

$$dv_t = \kappa(\theta - v_t)\, dt + \xi\sqrt{v_t}\, dW_t^v$$

**Price:**

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa_J)\, dt + \sqrt{v_t}\, dW_t^S + (e^J - 1)\, dN_t$$

**Correlation:**

$$\text{Corr}(dW_t^S, dW_t^v) = \rho$$

(usually negative: price down → vol up). Instantaneous volatility $= \sqrt{v_t}$.


## 1b. Simulation equations (not differential — what the code uses)

These are the discrete updates used in Monte Carlo (same role as $S_{t+\Delta t}=S_t\exp(\ldots)$ in GBM / Merton).

**Variance update:**

$$v_{t+\Delta t} = v_t + \kappa(\theta - v_t)\Delta t + \xi\sqrt{v_t}\sqrt{\Delta t}\, Z_v$$

In code we also clip $v \leftarrow \max(v, 0)$ so variance stays non-negative.

**Price update:**

$$S_{t+\Delta t} = S_t \exp\left( \left(\mu - \lambda\kappa_J - \frac{1}{2}v_t\right)\Delta t + \sqrt{v_t}\sqrt{\Delta t}\, Z_S + \sum_{i=1}^{N_{\Delta t}} J_i \right)$$

with
- $Z_S, Z_v \sim N(0,1)$ and $\text{Corr}(Z_S, Z_v)=\rho$
- $N_{\Delta t} \sim \text{Poisson}(\lambda\Delta t)$
- $J_i \sim N(\mu_J, \sigma_J^2)$ (if $N_{\Delta t}=0$, the sum is 0)

**Jump compensation** (derived from jump-size params, not a free slider):

$$\kappa_J = E[e^J - 1] = e^{\mu_J + \sigma_J^2/2} - 1$$


## 1c. Parameters (symbols)

| Symbol | Name | Meaning | Real world |
|--------|------|---------|------------|
| $S_t$ | price | stock price at time $t$ | what you see on a chart |
| $v_t$ | variance | vol squared at time $t$ | market stress level |
| $v_0$ | start variance | variance today | $v_0=0.04$ → about 20% vol |
| $\theta$ | long-run variance | level $v_t$ reverts to | normal long-run vol$^2$ |
| $\kappa$ | mean reversion speed | how fast $v_t$ returns to $\theta$ | low $\kappa$ = vol stays high for months |
| $\xi$ | vol-of-vol | randomness in variance itself | high $\xi$ = vol can spike hard |
| $\rho$ | correlation | price shock vs vol shock | usually **negative** (leverage effect) |
| $\mu$ | drift | average annual trend | same idea as GBM |
| $\lambda$ | jump intensity | expected jumps per year | same as Merton |
| $\mu_J$ | mean log jump | typical jump size | e.g. $-0.05$ ≈ −5% |
| $\sigma_J$ | jump size vol | uncertainty of jump size | larger → wilder jumps |
| $\kappa_J$ | jump compensation | $e^{\mu_J+\sigma_J^2/2}-1$ | keeps mean drift consistent |
| $\Delta t$ | time step | years per step | often $1/252$ |

Default playground values (same spirit as the handbook): $\kappa=2$, $\theta=0.04$, $\xi=0.5$, $\rho=-0.6$, $\lambda=0.3$, $\mu_J=-0.05$, $\sigma_J=0.10$.


## 2. End-to-end workflow (first time → Monte Carlo paths)

Do these steps in order. Steps 1–6 use historical data / design choices once (or once per regime). Step 7 repeats inside the simulation.

| Step | What you do | Output |
|------|-------------|--------|
| **1. Collect prices** | Get adjusted closes for your ticker (e.g. SPY) over a window / regime | Price series $S_0,S_1,\ldots$ |
| **2. Compute log returns** | $r_t = \ln(S_t / S_{t-1})$ | Return series |
| **3. Estimate Heston vol params** | From returns / realized variance: $\theta, v_0, \kappa, \xi, \rho$ | Vol block |
| **4. Estimate jump params** | Same spirit as Merton: $\lambda, \mu_J, \sigma_J$ → then $\kappa_J$ | Jump block |
| **5. Estimate / choose drift $\mu$** | Mean of returns (annualized), or risk-neutral drift later for pricing | $\mu$ |
| **6. Set simulation design** | Choose $S_0$, horizon $T$, steps $n$, number of paths | Grid: $\Delta t = T/n$ |
| **7. Simulate paths** | Each step update **both** $v_t$ and $S_t$ (correlated shocks + jumps) | Cloud of prices **and** vols |
| **8. Use the paths** | Histograms, option payoffs $\max(S_T-K,0)$, American exercise later | Research outputs |

**Regimes in this project** (fit separately if you want regime-specific parameters):

| Regime | Window | Typical Heston–Merton pattern |
|--------|--------|-------------------------------|
| Crisis | 2007–2009 | higher $\theta$, higher $\xi$, more negative jumps |
| Normal | 2013–2014 | lower vol level |
| Late | 2018–2019 | mixed / elevated vol |
| COVID | 2019–2020 | jump / fat-tail window |

Heston has more parameters than GBM — start with rough historical estimates, refine later (e.g. options calibration).


## 3. How to calculate parameters from historical data

Practical **moment / proxy** recipes (good enough to understand the pipeline). Advanced MLE / particle filters / option calibration can come later.

Assume daily returns and $N_{\text{days}}=252$ trading days per year. Data: `../data/equity/log_returns_by_regime.csv`.

### Step A — log returns

$$r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right)$$

### Step B — variance proxy from returns

$$RV_t = r_t^2 \quad\text{or}\quad RV_t = \frac{1}{w}\sum_{i=0}^{w-1} r_{t-i}^2$$

Annualize when comparing to yearly $\theta$: daily variance $\times 252$.

### Step C — long-run variance $\theta$ (estimated once)

$$\hat\theta_{\text{daily}} = \overline{RV},\qquad \hat\theta = \hat\theta_{\text{daily}}\times 252$$

Rule of thumb: if long-run vol ≈ 20%, then $\theta \approx 0.20^2 = 0.04$.

### Step D — starting variance $v_0$ (once per simulation start)

$$\hat v_0 = RV_{\text{recent}}\times 252$$

### Step E — mean-reversion speed $\kappa$ (estimated once)

If $\mathrm{Corr}(RV_t, RV_{t+1})\approx \rho_1$, a rough AR(1) link is $\rho_1 \approx e^{-\kappa\Delta t}$, so

$$\hat\kappa \approx -\frac{\ln(\rho_1)}{\Delta t}$$

(with $\Delta t=1/252$). Low $\kappa$ ⇒ vol stays high for a long time after a shock.

### Step F — vol-of-vol $\xi$ (estimated once)

From variability of the variance proxy (method-of-moments on $\Delta RV$). Larger $\xi$ ⇒ wilder vol spikes. Typical playground values: $\xi \in [0.3, 1.0]$.

### Step G — correlation $\rho$ (estimated once)

$$\hat\rho = \mathrm{Corr}\big(r_t,\ \Delta RV_t\big)$$

Equity index: usually **negative** (leverage effect), e.g. $\rho \approx -0.6$.

### Step H — jump block (estimated once)

Same as [`02_merton.ipynb`](02_merton.ipynb): threshold large returns → $\hat\lambda, \hat\mu_J, \hat\sigma_J$, then

$$\kappa_J = e^{\hat\mu_J + \hat\sigma_J^2/2} - 1$$

### Step I — drift $\mu$ (estimated once)

$$\hat\mu = \bar{r}\times 252$$

### Quick Python sketch (not run here — for later calibration)

```python
import pandas as pd
import numpy as np

r = pd.read_csv("../data/equity/log_returns_by_regime.csv")
sp = r[(r["ticker"]=="SPY") & (r["regime"]=="normal")]["log_return"]
mu_hat = sp.mean() * 252
rv = sp**2
theta_hat = rv.mean() * 252          # rough long-run variance
v0_hat = rv.iloc[-21:].mean() * 252  # recent window
rho1 = rv.autocorr(lag=1)
kappa_hat = -np.log(max(rho1, 1e-6)) * 252
```


## 4. Which parameters are constant vs which change along a path?

This is the key distinction for Monte Carlo (same section as in notebooks 01, 02, and 04).

### Calibrated once (constant for the whole simulation)

These are estimated from history **before** you simulate. They do **not** update step-by-step inside a path.

| Parameter | Role | Updates during a path? |
|-----------|------|------------------------|
| $\mu$ | drift | **No** — fixed |
| $\kappa,\theta,\xi,\rho$ | Heston vol dynamics | **No** — fixed rules |
| $v_0$ | initial variance | **No** — starting value only |
| $\lambda,\mu_J,\sigma_J,\kappa_J$ | jumps | **No** — fixed |
| $S_0$, $T$, $\Delta t$, $n_{\text{paths}}$ | design choices | **No** |

### Evolve along each Monte Carlo path

| Quantity | Role | Updates during a path? |
|----------|------|------------------------|
| $v_t$ | variance state | **Yes** — random, mean-reverting path |
| $S_t$ | price | **Yes** — uses current $\sqrt{v_t}$ |
| $Z_S, Z_v$ | correlated shocks | **Yes** — new draws every step |
| $N_{\Delta t}, J_i$ | jumps | **Yes** — redrawn every step |

**Big difference from GBM / constant-vol Merton:** the Monte Carlo state is the pair $(S_t, v_t)$, not price alone.

### One simulation step (repeat for $t = 0, \Delta t, 2\Delta t, \ldots$)

1. Keep $(\mu,\kappa,\theta,\xi,\rho,v_0,\lambda,\mu_J,\sigma_J)$ as the same numbers you estimated.
2. Draw correlated $Z_S, Z_v$ with correlation $\rho$.
3. Update variance: $v_{t+\Delta t} = v_t + \kappa(\theta-v_t)\Delta t + \xi\sqrt{v_t}\sqrt{\Delta t}\,Z_v$, then clip at 0.
4. Draw $N\sim\mathrm{Poisson}(\lambda\Delta t)$ and jump sizes if needed.
5. Update price with the exponential formula in section 1b, using **current** $v_t$.


## 5. Interactive playground

Watch the **price** panel and the **instantaneous vol** $\sqrt{v_t}$ panel together.

- Raise $\xi$ → wilder vol paths
- Make $\rho$ more negative → stronger leverage effect (price down, vol up)
- Raise $\lambda$ or $\sigma_J$ → more / larger jumps on top of stochastic vol

When you later calibrate to SPY (or another ticker), plug your estimated parameters into these same sliders.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_heston_merton(
    mu, kappa, theta, xi, rho, v0,
    lam, mu_j, sigma_j,
    S0, T, n_steps, n_paths, seed=42,
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    kappa_j = np.exp(mu_j + 0.5 * sigma_j**2) - 1.0

    S = np.full(n_paths, S0, dtype=float)
    v = np.full(n_paths, v0, dtype=float)
    paths = np.empty((n_paths, n_steps + 1))
    vol_paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = S
    vol_paths[:, 0] = np.sqrt(np.maximum(v, 0.0))
    log_rets = np.empty((n_paths, n_steps))

    for i in range(n_steps):
        z1 = rng.standard_normal(n_paths)
        z2 = rng.standard_normal(n_paths)
        dWs = z1
        dWv = rho * z1 + np.sqrt(max(1.0 - rho**2, 0.0)) * z2

        v_pos = np.maximum(v, 0.0)
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * dWv
        v = np.maximum(v, 0.0)

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump = np.zeros(n_paths)
        mask = n_jumps > 0
        jump[mask] = (
            n_jumps[mask] * mu_j
            + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
        )

        incr = (mu - 0.5 * v_pos - lam * kappa_j) * dt + np.sqrt(v_pos * dt) * dWs + jump
        S = S * np.exp(incr)
        paths[:, i + 1] = S
        vol_paths[:, i + 1] = np.sqrt(v)
        log_rets[:, i] = incr

    t = np.linspace(0, T, n_steps + 1)
    return t, paths, vol_paths, log_rets

def plot_heston_merton(
    mu=0.05, kappa=2.0, theta=0.04, xi=0.5, rho=-0.6, v0=0.04,
    lam=0.3, mu_j=-0.05, sigma_j=0.10,
    S0=100.0, T=1.0, n_steps=500, n_paths=1000,
):
    t, paths, vol_paths, log_rets = simulate_heston_merton(
        mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.8)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean")
    axes[0].set_title("Price paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].plot(t, vol_paths.T, alpha=0.35, lw=0.8)
    axes[1].plot(t, vol_paths.mean(axis=0), color="black", lw=2)
    axes[1].set_title("Instantaneous vol √v")
    axes[1].set_xlabel("years")

    axes[2].hist(log_rets.ravel(), bins=80, density=True, alpha=0.75, color="seagreen")
    axes[2].set_title("Step log returns")
    axes[2].set_xlabel("log return")

    fig.suptitle(
        f"κ={kappa:.1f}, θ={theta:.3f}, ξ={xi:.2f}, ρ={rho:.2f}, λ={lam:.2f}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_heston_merton,
    mu=FloatSlider(value=0.05, min=-0.10, max=0.30, step=0.01, description="μ"),
    kappa=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description="κ"),
    theta=FloatSlider(value=0.04, min=0.005, max=0.20, step=0.005, description="θ var"),
    xi=FloatSlider(value=0.50, min=0.05, max=2.0, step=0.05, description="ξ volvol"),
    rho=FloatSlider(value=-0.60, min=-0.95, max=0.95, step=0.05, description="ρ"),
    v0=FloatSlider(value=0.04, min=0.005, max=0.20, step=0.005, description="v0"),
    lam=FloatSlider(value=0.30, min=0.0, max=3.0, step=0.1, description="λ"),
    mu_j=FloatSlider(value=-0.05, min=-0.30, max=0.15, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.10, min=0.01, max=0.40, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="T"),
    n_steps=IntSlider(value=500, min=50, max=2000, step=10, description="steps"),
    n_paths=IntSlider(value=1000, min=5, max=2000, step=5, description="paths"),
);